# 

In [2]:
import pyspark
from pyspark.sql import SparkSession
spark = SparkSession.builder \
        .master("local[*]") \
        .appName('test') \
        .config('spark.driver.memory', '8g') \
        .getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/06 10:18:49 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
df_green = spark.read \
    .option("recursiveFileLookup", "true") \
    .parquet('data/pq/green/')

In [4]:
df_green.createOrReplaceTempView("green")

In [5]:
df_green_revenue = spark.sql("""
SELECT 
    -- Revenue grouping 
    date_trunc('hour', lpep_pickup_datetime) AS hour, 
    PULocationID AS zone,

    -- Revenue calculation 
    SUM(total_amount) AS amount,
    COUNT(1) AS number_records
FROM
    green
WHERE
    lpep_pickup_datetime >= '2020-01-01 00:00:00'
GROUP BY
    1, 2
ORDER BY
    1, 2
""")

In [8]:
df_green_revenue.write.parquet('data/report/revenue/green/', mode = 'overwrite')

In [9]:
df_yellow = spark.read \
    .option("recursiveFileLookup", "true") \
    .parquet('data/pq/yellow/')

In [10]:
df_yellow.createOrReplaceTempView("yellow")

In [11]:
df_yellow_revenue = spark.sql("""
SELECT 
    -- Revenue grouping 
    date_trunc('hour', tpep_pickup_datetime) AS hour, 
    PULocationID AS zone,

    -- Revenue calculation 
    SUM(total_amount) AS amount,
    COUNT(1) AS number_records
FROM
    yellow
WHERE
    tpep_pickup_datetime >= '2020-01-01 00:00:00'
GROUP BY
    1, 2
ORDER BY
    1, 2
""")

In [12]:
df_yellow_revenue.repartition(20).write.parquet('data/report/revenue/yellow/', mode="overwrite")

In [13]:
df_green_revenue.repartition(20).write.parquet('data/report/revenue/green/', mode="overwrite")

In [18]:
df_green_revenue = spark.read.parquet('data/report/revenue/green')
df_yellow_revenue = spark.read.parquet('data/report/revenue/yellow')

In [19]:
df_green_tmp = df_green_revenue \
    .withColumnRenamed('amount', 'green_amount') \
    .withColumnRenamed('number_records', 'green_number_records')

In [20]:
df_yellow_tmp = df_yellow_revenue \
    .withColumnRenamed('amount', 'yellow_amount') \
    .withColumnRenamed('number_records', 'yellow_number_records')

In [23]:
df_join = df_green_tmp.join(df_yellow_tmp, on=['hour', 'zone'], how='outer')

In [24]:
df_join.write.parquet('data/report/revenue/total', mode='overwrite')

In [25]:
df_join.show()

+-------------------+----+------------------+--------------------+------------------+---------------------+
|               hour|zone|      green_amount|green_number_records|     yellow_amount|yellow_number_records|
+-------------------+----+------------------+--------------------+------------------+---------------------+
|2020-01-01 00:00:00|  51|              17.8|                   2|              31.0|                    1|
|2020-01-01 00:00:00|  71|              23.8|                   1|              NULL|                 NULL|
|2020-01-01 00:00:00| 136|            111.68|                   2|             168.0|                    4|
|2020-01-01 00:00:00| 143|             55.95|                   2|2142.0600000000004|                  128|
|2020-01-01 00:00:00| 167|             88.89|                   5|               9.8|                    1|
|2020-01-01 00:00:00| 197|             72.01|                   3|              NULL|                 NULL|
|2020-01-01 00:00:00| 211|  

In [29]:
df_join = spark.read.parquet('data/report/revenue/total')

In [32]:
!wget https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv

--2026-03-06 10:28:00--  https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv
Resolving d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)... 2600:9000:2462:ac00:b:20a5:b140:21, 2600:9000:2462:4e00:b:20a5:b140:21, 2600:9000:2462:1a00:b:20a5:b140:21, ...
Connecting to d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)|2600:9000:2462:ac00:b:20a5:b140:21|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 12331 (12K) [text/csv]
Saving to: ‘taxi_zone_lookup.csv’

taxi_zone_lookup.cs 100%[===================>]  12.04K  --.-KB/s    in 0s      

2026-03-06 10:28:00 (1.64 GB/s) - ‘taxi_zone_lookup.csv’ saved [12331/12331]



In [33]:
df = spark.read \
    .option("header", "true") \
    .csv('taxi_zone_lookup.csv')

df.show()

+----------+-------------+--------------------+------------+
|LocationID|      Borough|                Zone|service_zone|
+----------+-------------+--------------------+------------+
|         1|          EWR|      Newark Airport|         EWR|
|         2|       Queens|         Jamaica Bay|   Boro Zone|
|         3|        Bronx|Allerton/Pelham G...|   Boro Zone|
|         4|    Manhattan|       Alphabet City| Yellow Zone|
|         5|Staten Island|       Arden Heights|   Boro Zone|
|         6|Staten Island|Arrochar/Fort Wad...|   Boro Zone|
|         7|       Queens|             Astoria|   Boro Zone|
|         8|       Queens|        Astoria Park|   Boro Zone|
|         9|       Queens|          Auburndale|   Boro Zone|
|        10|       Queens|        Baisley Park|   Boro Zone|
|        11|     Brooklyn|          Bath Beach|   Boro Zone|
|        12|    Manhattan|        Battery Park| Yellow Zone|
|        13|    Manhattan|   Battery Park City| Yellow Zone|
|        14|     Brookly

In [34]:
df.write.parquet('zones')

In [27]:
df_join.show()

+-------------------+----+------------------+--------------------+------------------+---------------------+
|               hour|zone|      green_amount|green_number_records|     yellow_amount|yellow_number_records|
+-------------------+----+------------------+--------------------+------------------+---------------------+
|2020-01-01 00:00:00|  17|195.03000000000003|                   9|220.21000000000004|                    8|
|2020-01-01 00:00:00|  34|              NULL|                NULL|              19.3|                    1|
|2020-01-01 00:00:00|  36|295.34000000000003|                  11|            109.17|                    3|
|2020-01-01 00:00:00|  52|             83.33|                   4|              49.8|                    2|
|2020-01-01 00:00:00|  60|160.04000000000002|                   6|57.620000000000005|                    2|
|2020-01-01 00:00:00| 106|             10.56|                   1|              NULL|                 NULL|
|2020-01-01 00:00:00| 112|31

In [35]:
df_zones = spark.read.parquet('zones/')

In [52]:
df_result = df_join.join(df_zones, df_join.zone == df_zones.LocationID).drop('LocationID', 'zone')


In [53]:
df_result.write.parquet('tmp/revenue-zones')

In [55]:
df_result.explain(mode="formatted")

== Physical Plan ==
AdaptiveSparkPlan (8)
+- Project (7)
   +- BroadcastHashJoin Inner BuildRight (6)
      :- Filter (2)
      :  +- Scan parquet  (1)
      +- BroadcastExchange (5)
         +- Filter (4)
            +- Scan parquet  (3)


(1) Scan parquet 
Output [6]: [hour#213, zone#214, green_amount#215, green_number_records#216L, yellow_amount#217, yellow_number_records#218L]
Batched: true
Location: InMemoryFileIndex [file:/home/peter/repos/data-engineering-zoomcamp/06-batch/data/report/revenue/total]
PushedFilters: [IsNotNull(zone)]
ReadSchema: struct<hour:timestamp,zone:int,green_amount:double,green_number_records:bigint,yellow_amount:double,yellow_number_records:bigint>

(2) Filter
Input [6]: [hour#213, zone#214, green_amount#215, green_number_records#216L, yellow_amount#217, yellow_number_records#218L]
Condition : isnotnull(zone#214)

(3) Scan parquet 
Output [3]: [LocationID#262, Borough#263, service_zone#265]
Batched: true
Location: InMemoryFileIndex [file:/home/peter/repos/